# Módulo 2 — Dia 1
## Atividades extras: Sistemas de Equações Lineares

Três atividades complementares às da aula, cada uma focada em uma das três abordagens vistas:
**eliminação de Gauss** (manual), **decomposição LU** (programação) e **métodos iterativos**
(escolha entre Jacobi ou Gauss-Seidel).

São problemas **novos**, diferentes dos exemplos trabalhados em aula — resolva-os por conta
própria, na ordem, antes de conferir com as validações automáticas.

> Dica: execute as células na ordem, de cima para baixo (Shift+Enter).

---
## Atividade 1 — Eliminação de Gauss, manual (matriz 5×5)

Resolva **inteiramente no papel**, por eliminação de Gauss (escalonamento + substituição
retroativa), o sistema $Ax = b$ abaixo:

$$
\begin{pmatrix}
4 & -1 & 0 & 1 & 0 \\
-1 & 4 & -1 & 0 & 1 \\
0 & -1 & 4 & -1 & 0 \\
1 & 0 & -1 & 4 & -1 \\
0 & 1 & 0 & -1 & 4
\end{pmatrix}
\begin{pmatrix} x_1\\x_2\\x_3\\x_4\\x_5 \end{pmatrix}
=
\begin{pmatrix} 6\\9\\6\\9\\18 \end{pmatrix}
$$

**Roteiro sugerido:**
1. Monte a matriz aumentada $[A \mid b]$.
2. Escalone até obter uma matriz triangular superior (mostre cada etapa — qual linha foi usada
   como pivô e qual múltiplo foi subtraído de qual linha).
3. Resolva por substituição retroativa, de $x_5$ até $x_1$.
4. Preencha sua resposta na célula markdown abaixo antes de rodar a validação.

*(A matriz foi escolhida para que a solução tenha apenas números inteiros pequenos — se sua
resposta não bater com números "redondos", revise as contas antes de seguir.)*

**Minha resposta manual:**

x₁ = ____   x₂ = ____   x₃ = ____   x₄ = ____   x₅ = ____

*(preencha antes de rodar a célula de validação abaixo)*

In [1]:
import numpy as np

A1 = np.array([
    [4, -1, 0, 1, 0],
    [-1, 4, -1, 0, 1],
    [0, -1, 4, -1, 0],
    [1, 0, -1, 4, -1],
    [0, 1, 0, -1, 4],
], dtype=float)
b1 = np.array([6, 9, 6, 9, 18], dtype=float)

# TODO: digite aqui sua resposta manual, na ordem [x1, x2, x3, x4, x5]
resposta_manual = np.array([1, 2, 3, 4, 5])

# Conferência automática (reaproveitando a função de eliminação de Gauss do Dia 1)
def eliminacao_de_gauss(A, b):
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = len(b)
    for k in range(n - 1):
        pivo = np.argmax(np.abs(A[k:, k])) + k
        A[[k, pivo]] = A[[pivo, k]]
        b[[k, pivo]] = b[[pivo, k]]
        for i in range(k + 1, n):
            fator = A[i, k] / A[k, k]
            A[i, k:] -= fator * A[k, k:]
            b[i] -= fator * b[k]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
    return x

x1_codigo = eliminacao_de_gauss(A1, b1)
print("Solução por código (Gauss):", x1_codigo)
print("Sua resposta manual:       ", resposta_manual)


Solução por código (Gauss): [1. 2. 3. 4. 5.]
Sua resposta manual:        [1 2 3 4 5]


In [2]:
# Validação
assert np.allclose(resposta_manual, x1_codigo, atol=1e-6), \
    "Sua resposta manual não bate com o código — revise o escalonamento e a substituição retroativa"
print("Validação OK! A solução do sistema é x =", np.round(x1_codigo).astype(int))


Validação OK! A solução do sistema é x = [1 2 3 4 5]


---
## Atividade 2 — Decomposição LU, com programação

**Problema:** uma estrutura simples com 4 nós livres tem matriz de rigidez $K$ (fixa — depende só
da geometria e do material). Três cenários de carregamento diferentes (peso próprio, vento e
sobrecarga) produzem três vetores de força $F$ diferentes. Calcule o deslocamento de cada nó
($Kd = F$) nos três cenários.

Como $K$ é a mesma nos três casos, este é exatamente o tipo de situação em que decompor $K = LU$
**uma única vez** e reaproveitar o fatoramento é mais eficiente do que resolver o sistema do zero
a cada novo vetor $F$.

$$
K = \begin{pmatrix}
6 & -2 & 0 & 0 \\
-2 & 7 & -3 & 0 \\
0 & -3 & 8 & -4 \\
0 & 0 & -4 & 9
\end{pmatrix}
$$

$$F_{peso} = (10, 15, 12, 8) \qquad F_{vento} = (5, -5, 5, -5) \qquad F_{sobrecarga} = (20, 20, 20, 20)$$

Complete o código abaixo (não resolva à mão desta vez — o ponto da atividade é praticar a
decomposição LU em código):

In [3]:
import numpy as np
from scipy.linalg import lu_factor, lu_solve

K = np.array([
    [6, -2, 0, 0],
    [-2, 7, -3, 0],
    [0, -3, 8, -4],
    [0, 0, -4, 9],
], dtype=float)

F_peso = np.array([10, 15, 12, 8], dtype=float)
F_vento = np.array([5, -5, 5, -5], dtype=float)
F_sobrecarga = np.array([20, 20, 20, 20], dtype=float)

# TODO: decomponha K em LU uma única vez
lu, piv = lu_factor(K)

# TODO: resolva Kd=F para os três cenários, reaproveitando (lu, piv)
d_peso = lu_solve((lu, piv), F_peso)
d_vento = lu_solve((lu, piv), F_vento)
d_sobrecarga = lu_solve((lu, piv), F_sobrecarga)

for nome, d in [("peso próprio", d_peso), ("vento", d_vento), ("sobrecarga", d_sobrecarga)]:
    print(f"Deslocamentos ({nome}): {np.round(d, 4)}")


Deslocamentos (peso próprio): [3.4287 5.2862 5.0487 3.1328]
Deslocamentos (vento): [ 0.6973 -0.408   0.2497 -0.4446]
Deslocamentos (sobrecarga): [6.1023 8.3069 8.648  6.0658]


### Pergunta

Se você tivesse resolvido os três cenários com `np.linalg.solve(K, F)` chamado três vezes (em vez
de reaproveitar `lu_factor`/`lu_solve`), o resultado numérico seria diferente? E o custo
computacional? Responda em 1–2 linhas.

*Não, os resultados numéricos seriam os mesmos. Porém, enquanto o np.linalg.solve(K, F) que recalcula a fatoração da matriz \(K\) a cada chamada aumenta o custo computacional, o lu_factor realiza essa fatoração apenas uma vez, e o lu_solve a reutiliza para todos os vetores de força, o que torna o processo mais eficiente.*

In [4]:
# Validação
assert np.allclose(K @ d_peso, F_peso, atol=1e-8), "d_peso não satisfaz K@d = F"
assert np.allclose(K @ d_vento, F_vento, atol=1e-8), "d_vento não satisfaz K@d = F"
assert np.allclose(K @ d_sobrecarga, F_sobrecarga, atol=1e-8), "d_sobrecarga não satisfaz K@d = F"

# Confere que reaproveitar (lu, piv) dá o mesmo resultado que resolver do zero
assert np.allclose(d_peso, np.linalg.solve(K, F_peso), atol=1e-8)
assert np.allclose(d_vento, np.linalg.solve(K, F_vento), atol=1e-8)
assert np.allclose(d_sobrecarga, np.linalg.solve(K, F_sobrecarga), atol=1e-8)
print("Validação OK! Os três cenários foram resolvidos corretamente reaproveitando a LU.")


Validação OK! Os três cenários foram resolvidos corretamente reaproveitando a LU.


---
## Atividade 3 — Jacobi **ou** Gauss-Seidel (escolha um dos dois)

**Problema:** distribuição de temperatura em regime permanente ao longo de uma barra 1D com 5
pontos internos, discretizada por diferenças finitas. Cada nó $i$ satisfaz um balanço de energia
que, após embutir as temperaturas de contorno, resulta no sistema tridiagonal abaixo (matriz
diagonalmente dominante — condição que garante a convergência dos dois métodos):

$$
\begin{pmatrix}
2.2 & -1 & 0 & 0 & 0 \\
-1 & 2.2 & -1 & 0 & 0 \\
0 & -1 & 2.2 & -1 & 0 \\
0 & 0 & -1 & 2.2 & -1 \\
0 & 0 & 0 & -1 & 2.2
\end{pmatrix}
\begin{pmatrix} T_1\\T_2\\T_3\\T_4\\T_5 \end{pmatrix}
=
\begin{pmatrix} 20\\0\\0\\0\\30 \end{pmatrix}
$$

**Escolha um dos dois métodos (Jacobi ou Gauss-Seidel) e complete apenas a função
correspondente** na célula abaixo — não precisa implementar os dois. Parta de $T^{(0)} = (0,0,0,0,0)$.

In [5]:
A3 = np.zeros((5, 5))
for i in range(5):
    A3[i, i] = 2.2
    if i > 0:
        A3[i, i - 1] = -1
    if i < 4:
        A3[i, i + 1] = -1
b3 = np.array([20, 0, 0, 0, 30], dtype=float)

metodo_escolhido = "gauss_seidel"  # TODO: troque para "gauss_seidel" se preferir esse método


def jacobi(A, b, x0, tol=1e-6, max_iter=300):
    x = x0.astype(float).copy()
    D = np.diag(A)
    R = A - np.diagflat(D)
    for k in range(max_iter):
        # TODO: complete a atualização de Jacobi (se for o método escolhido)
        x_novo = (b - R @ x) / D
        if np.linalg.norm(x_novo - x) < tol:
            return x_novo, k + 1
        x = x_novo
    return x, max_iter


def gauss_seidel(A, b, x0, tol=1e-6, max_iter=300):
    x = x0.astype(float).copy()
    n = len(b)
    for k in range(max_iter):
        x_antigo = x.copy()
        for i in range(n):
            # TODO: complete a atualização de Gauss-Seidel (se for o método escolhido)
            soma = sum(A[i, j] * x[j] for j in range(n) if j != i)
            x[i] = (b[i] - soma) / A[i, i]
        if np.linalg.norm(x - x_antigo) < tol:
            return x, k + 1
    return x, max_iter


if metodo_escolhido == "jacobi":
    T, n_iter = jacobi(A3, b3, np.zeros(5))
else:
    T, n_iter = gauss_seidel(A3, b3, np.zeros(5))

print(f"Método escolhido: {metodo_escolhido}")
print(f"Temperaturas convergidas: {np.round(T, 4)}")
print(f"Número de iterações até convergir: {n_iter}")


Método escolhido: gauss_seidel
Temperaturas convergidas: [14.6749 12.2849 12.3518 14.889  20.4041]
Número de iterações até convergir: 36


### Desafio (opcional)

Implemente também o outro método (o que você não escolheu) e compare o número de iterações até
convergir. Qual dos dois converge mais rápido neste sistema?

In [ ]:
# Desafio opcional: rode o outro método aqui e compare
T_jacobi, n_jacobi = jacobi(A3, b3, np.zeros(5))
T_gs, n_gs = gauss_seidel(A3, b3, np.zeros(5))
print(f"Jacobi convergiu em {n_jacobi} iterações")
print(f"Gauss-Seidel convergiu em {n_gs} iterações")


Jacobi convergiu em 69 iterações
Gauss-Seidel convergiu em 36 iterações


In [ ]:
# Validação
T_exato = np.linalg.solve(A3, b3)
assert np.allclose(T, T_exato, atol=1e-4), "resultado do método escolhido não confere com a solução exata"
assert n_iter < 300, "o método não convergiu dentro do limite de iterações"
print("Validação OK! O método escolhido convergiu para a solução correta.")


Validação OK! O método escolhido convergiu para a solução correta.


---
### Não esqueça

Ao terminar as três atividades, faça commit e push deste notebook para o repositório GitHub:

```bash
git add dia1_atividades_extras_modulo2.ipynb
git commit -m "Módulo 2 - Dia 1: atividades extras (Gauss manual, LU e método iterativo)"
git push
```